# 36 · Security and tenancy for an LLM endpoint
Level L5 · Part 5 · core · ~20 min · builds on 33 (uses 14 and 29)

Time: about 20 min. This is a **concept** lesson: it explains the risks, then runs three short offline cells (no server, no Docker, no key) that make the points concrete. Every number and name in the examples is made up.

## Goal
By the end you can name the endpoints that an API key does not protect, explain why a per-tenant `cache_salt` stops one customer from learning what another customer sent (and why it is still not a full isolation boundary), and design a log line that keeps the audit trail without storing the prompt.

## Why it matters
- **Without it:** a team sets `--api-key`, marks the server "secured" and stops. The key checks only the paths under `/v1`, `/v2`, `/inference` and `/cohere`; the metrics page and other paths answer anyone who can reach the port. Two customers share one engine, and the prefix cache lets one measure how fast the other's prompt was answered. Full prompts go into logs that are copied, kept for years and read by people who never needed them.
- **With it:** a proxy allowlist exposes only the API paths, the gateway gives each tenant its own salt, and the log holds counts, ids and a keyed hash.
- **The cost of skipping it:** a leaked prompt is a data breach, and the fix after the fact is a customer notice, not a config change.
- **When you don't need it:** one engineer on one laptop with test prompts. Once a second person or customer shares the server, this applies.

## The idea
An LLM endpoint has three separate risks, and each needs its own control. **Doors:** the server exposes more paths than the API, and a key only guards the paths it is told to guard. **Shared memory:** the prefix cache from lesson 14 is shared between requests, so reuse is visible through timing; a per-tenant salt splits the cache. **Records:** logs copy whatever you let them see, so the prompt is a record of customer data unless you choose what to write. Think of an office building. The badge reader guards the main door only. A side door for the mail room has no lock. The shared kitchen fridge shows, by how empty it is, what someone else has eaten. And the visitor book copies every passport.

## Picture
```mermaid
flowchart LR
  C["Client with a tenant key"] --> P["Gateway proxy: allowlist, key check, sets cache_salt per tenant"]
  P -->|"/v1 paths only"| E["vLLM engine, one shared prefix cache, split by salt"]
  X["Other paths, e.g. /metrics"] -. "not covered by --api-key: block at the proxy" .-> P
  E --> L["Log: ids, token counts, model version, keyed prompt hash. No raw prompt"]
```

## Explained

### 1. Which doors the key does not guard
The vLLM security guidance says `--api-key` covers only the paths under `/v1`, `/v2`, `/inference` and `/cohere`. The same page lists utility endpoints that need no key, and `/metrics` is one of them. Other paths on the server are not under those prefixes either, so the key is not checked on them. Two more rules: `VLLM_SERVER_DEV_MODE=1` exposes a remote-procedure endpoint (`/collective_rpc`) that must never run in production, and profiler endpoints stay off.

The fix is not a longer key. Put a reverse proxy in front of the engine that forwards only the paths you have named, for example `/v1/chat/completions` and `/v1/models`, and bind the engine to a private address. Probes usually call `/health` without a key, so let it through only from the private network. Check the list for your vLLM version: the paths change between releases.

### 2. The prefix cache is a timing side channel
Lesson 14 showed that a block's hash chains over the block before it, so a request reuses the KV blocks of an identical prefix. Reused blocks skip computation, so the first token comes back sooner: timing reveals the reuse. vLLM's guidance calls this a timing side channel (CVE-2025-46570, a public ID for a security flaw).

The mitigation is `cache_salt`, a top-level request field mixed into the first block's hash; every later hash depends on it, so only equal salts share blocks. The **gateway** sets it from the authenticated tenant. It stores one stable, random, long salt per tenant, not a new one each run.

The docs add a limit: the salt is not an isolation boundary, because tenants on one process still share memory and the queue.

### 3. Logs: keep the count, not the words
A prompt holds customer text, names and addresses. A log that keeps it copies that data into every log store and backup, for the whole retention period. Log a record of **what happened** instead: request, tenant, key id (never the key), model version, token counts and latency. To match two requests with the same prompt, log a **keyed hash** (HMAC) of the prompt, with the key held by the log pipeline. A plain hash fails for short text: someone can hash guesses such as "yes" or an order number and compare. A keyed hash still matches equal prompts, but only the pipeline can test a guess. That record is the **audit trail**.

## Worked example: two tenants on one support desk
Brightside runs one vLLM engine for two customers: its own help desk (tenant A) and a partner shop (tenant B). Both send the same 1,500-token policy prompt first.

1. **Without a salt:** tenant B's first request finds tenant A's cached blocks and its first token arrives sooner. Tenant B learns that someone recently sent this policy text.
2. **With a per-tenant salt from the gateway:** tenant B's blocks hash differently. Its first token is as slow as a cold start.
3. **The log:** tenant B's line holds its tokens and a keyed hash of its question, not the address. An audit query can still answer "which key called `support-v3` on Tuesday, and how many tokens did it send?"

The cell below shows the key matching behind steps 1 and 2; the timing difference itself follows from lesson 14.

## Step by step
The next cells are a small demonstration. They do not call a server.

In [1]:
import hashlib, hmac, json, secrets

def first_block_key(salt, tokens):
    # A toy stand-in for the engine's block hash: the salt goes into the first block,
    # so every hash after it changes too. It is not vLLM's exact function.
    return hashlib.sha256(repr((salt, tokens[:16])).encode()).hexdigest()[:12]

policy = [f"rule{i}" for i in range(20)]     # stand-in for the shared policy prompt, as tokens
# The gateway stores one stable salt per tenant. This demo makes fresh ones only so they can be thrown away.
salt_a = secrets.token_hex(16)               # never printed
salt_b = secrets.token_hex(16)

tenant_a_req = first_block_key(None, policy)     # tenant A's request, no salt
tenant_b_req = first_block_key(None, policy)     # tenant B's request, no salt, same policy tokens
a_first = first_block_key(salt_a, policy)        # tenant A's first request
a_second = first_block_key(salt_a, policy)       # tenant A's second request, same salt
b_first = first_block_key(salt_b, policy)        # tenant B's request, its own salt
print("tenant A and tenant B, both unsalted, get the same key:", tenant_a_req == tenant_b_req)
print("tenant A's second request reuses its own key:          ", a_first == a_second)
print("tenant B's salted key matches tenant A's salted key:   ", b_first == a_first)

tenant A and tenant B, both unsalted, get the same key: True
tenant A's second request reuses its own key:           True
tenant B's salted key matches tenant A's salted key:    False


**What you should see.** Without a salt the two requests get the same key, so the second one reuses the first one's blocks. With a salt per tenant the keys differ, and the same tenant still gets its own reuse. The salts are never printed: a salt you can read is a salt someone else can use.

Next, the doors. The rule below is the one `--api-key` follows. Replace the sample paths with the routes of your own server.

In [2]:
COVERED = ("/v1/", "/v2/", "/inference/", "/cohere/")     # the prefixes --api-key checks
routes = ["/v1/chat/completions", "/v1/models", "/metrics", "/collective_rpc", "/health"]
notes = {"/collective_rpc": "(only exists if dev mode is on)",
         "/health": "(probes call it: private network only)"}
for route in routes:
    if route.startswith(COVERED):
        print(f"{route:<26} key checked")
    else:
        print(f"{route:<26} NO KEY: block at the proxy or keep private {notes.get(route, '')}".rstrip())

/v1/chat/completions       key checked
/v1/models                 key checked
/metrics                   NO KEY: block at the proxy or keep private
/collective_rpc            NO KEY: block at the proxy or keep private (only exists if dev mode is on)
/health                    NO KEY: block at the proxy or keep private (probes call it: private network only)


**What you should see.** Only the `/v1` route is checked by the key. Everything else needs a control in front of the engine: `/collective_rpc` should not exist at all outside dev mode, and `/health` belongs on the private network. Now the log. The record keeps everything an audit needs and replaces the prompt with a keyed hash.

In [3]:
LOG_KEY = secrets.token_bytes(32)     # in production: fixed, stored by the log pipeline; made fresh here, never printed

def prompt_hash(text):
    return hmac.new(LOG_KEY, text.encode(), hashlib.sha256).hexdigest()[:16]

def log_record(request_id, tenant, key_id, model_version, prompt, prompt_tokens, output_tokens, timestamp, status):
    return {
        "timestamp": timestamp,
        "request_id": request_id,
        "tenant": tenant,
        "key_id": key_id,                 # an id for the key, never the key itself
        "model_version": model_version,
        "status": status,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "prompt_hash": prompt_hash(prompt),
    }

prompt = "My order 4411 arrived broken. Reach me at name@example.com"
line = json.dumps(log_record("r-0001", "tenant-b", "key-07", "support-v3", prompt, 1512, 118,
                             "2026-10-20T09:14:02Z", "ok"))
print(line)
print("the raw prompt is in the line:", prompt in line, "| the address is in the line:", "name@example.com" in line)
print("same prompt, same hash:", prompt_hash(prompt) == prompt_hash(prompt))
print("different prompt, different hash:", prompt_hash(prompt) != prompt_hash("My order 4412 arrived broken."))

{"timestamp": "2026-10-20T09:14:02Z", "request_id": "r-0001", "tenant": "tenant-b", "key_id": "key-07", "model_version": "support-v3", "status": "ok", "prompt_tokens": 1512, "output_tokens": 118, "prompt_hash": "a743d3909ace87c8"}
the raw prompt is in the line: False | the address is in the line: False
same prompt, same hash: True
different prompt, different hash: True


**What you should see.** The line has what an auditor needs: timestamp, tenant, key id, model version, status, token counts and a 16-character hash. The address and the order text are not in it. The hash changes on every run of this demo, because the demo key is new each time. In production the key is fixed, so the same prompt gives the same hash across requests. A reader who has the log but not the key cannot test a guess against the hash.

## At work
- vLLM's guidance says it does not isolate tenants in one process and names the prefix-cache timing channel (CVE-2025-46570). Teams salt each customer's cache and give hard cases separate engines → lesson 29 (the gateway) and lesson 30 (routing).
- Gateways such as LiteLLM issue per-tenant virtual keys with limits and budgets; some features are paid tiers, so check each one. Lesson 29 builds the LiteLLM gateway; per-tenant virtual keys and budgets are a Go-deeper item.
- Engine-to-engine traffic (tensor, pipeline and KV-transfer) is not encrypted by vLLM. Keep it on a private network, firewall the `--kv-port`, and use mTLS if your rules require it. Lesson 34 introduces this traffic; it does not cover these controls.
- `/metrics` answers without a key, so keep it on the monitoring network → lesson 33 (the dashboard).

Go deeper: the vLLM security page lists more surfaces (multimodal URL fetching, LoRA loading, tool servers).

## Common misconceptions
1. **"`--api-key` protects the server."** It protects only the paths under `/v1`, `/v2`, `/inference` and `/cohere`. Metrics and other routes answer without it.
2. **"A cache salt makes tenants isolated."** It stops block sharing across salts and closes the timing signal. The docs say it is not an isolation boundary. Hard isolation means a separate engine.
3. **"Hashing the prompt makes the log anonymous."** A plain hash of a short text can be guessed by hashing candidates. Use a keyed HMAC, with the key outside the log.
4. **"We will add log rules later."** Logging libraries and proxies record by default. Decide what is written before the first customer sends a prompt.

## Production checklist
- Only the API paths are reachable from outside. Utility, control and profiler paths sit on a private network or behind the proxy allowlist.
- Dev mode and profiler endpoints are off in production.
- The gateway sets `cache_salt` per tenant from the authenticated identity. Each salt is random and long.
- Tenants with hard isolation needs have their own engine.
- Logs hold ids, token counts, model version, latency and a keyed prompt hash. Raw prompts are written only to a reviewed debug capture with an expiry date.
- Every request can be traced to a key id and a model version in the audit trail, and the retention period is set.
- Test it from outside: call every route without a key and check the answer.

## Check yourself
1. A scan reports that `/metrics` answers without a key. Is this a vLLM bug? What do you change, and what do you leave alone?
2. Tenant B says that the first token of its policy-based answers is consistently faster than a cold start, and it only happens after tenant A's traffic. Name the leak and give two fixes.
3. A client sends its own `cache_salt` in the request body. What can go wrong, and where should the salt come from?
4. A customer asks you to delete everything you hold about their prompts. Your logs hold full prompts for 400 days. What is the problem, and what log design would have made the answer easy?
5. An auditor asks which key called `support-v3` last Tuesday and how many tokens it sent. Which fields must the log hold, which must it never hold, and how long are the lines kept?

**Answers**
1. vLLM documents `/metrics` as unauthenticated: expected, not a bug. Fix the deployment: the proxy forwards only API paths; the scrape endpoint stays on the monitoring network.
2. A prefix-cache timing side channel (CVE-2025-46570). Fixes: a per-tenant, random `cache_salt` set by the gateway; for hard isolation, a separate engine per tenant.
3. The client can pick another tenant's salt and read that tenant's cache timing. The gateway sets the salt from the authenticated tenant and overwrites any client value.
4. Full prompts are copies in every log store and backup for 400 days, so deletion means hunting them all. Counts, ids and keyed hashes make the answer "nothing to delete except the capture".
5. Fields: timestamp, request id, tenant, key id, model version, prompt and output token counts, status. Never: the key, the raw prompt or the answer text. Delete the lines when the audit retention period ends.

## Recap
- **The key guards only the API paths.** Put a proxy allowlist in front and keep utility and control paths private.
- **The prefix cache leaks timing.** A per-tenant, random `cache_salt` set by the gateway stops the sharing, but it is not isolation: hard cases get their own engine.
- **Logs keep the count, not the words.** Ids, token counts, model version and a keyed hash give an audit trail without customer text.

**Interview:** "Two customers share one vLLM server. What do you lock down?" Strong answer: the gateway authenticates each customer and sets a random per-tenant `cache_salt`, only the API paths are exposed, the logs keep counts and keyed hashes, and any customer who needs hard isolation gets a separate engine, because vLLM does not isolate tenants in one process.

## Key terms
- **Allowlist (proxy)** — a rule that forwards only the paths you name and blocks everything else (lesson 36).
- **Audit trail** — a record of who called which model version, when, and with how many tokens, kept without the prompt text (lesson 36).
- **cache_salt** — a request field that is mixed into the first block's hash, so only requests with the same salt share prefix-cache blocks (lesson 36).
- **Keyed hash (HMAC)** — a hash that needs a secret key to recompute, so a log reader cannot test guesses (lesson 36).
- **Tenant** — one customer or team that shares a server and must not see another's data (lesson 36).
- **Timing side channel** — information leaked by how long an answer takes, here by a faster first token after a cache hit (lesson 36).